In [ ]:
from importlib import reload
import sys
sys.path.append("..")
import keypoint_detectors
reload(keypoint_detectors)
from keypoint_detectors import FastDetector, SuperPointDetector

import sys
from torch.utils.data import DataLoader
sys.path.append("..")
import conf


import core.QuadtreeWSIPairDataset
reload(core.QuadtreeWSIPairDataset)
reload(conf)
from core.QuadtreeWSIPairDataset import QuadtreeWSIPairDataset

ANNOTATION_PATH = conf.ANNOTATION_PATH
CNN_INPUT_HEIGHT = conf.CNN_INPUT_HEIGHT
CNN_INPUT_WIDTH = conf.CNN_INPUT_WIDTH


import torch
import matplotlib.pyplot as plt
import torch


def tensor_to_numpy_gray(t):
    # t: [1, H, W], float32 [0, 1]
    return t.squeeze(0).detach().cpu().numpy()

def tensor_to_numpy_rgb(t):
    # t: [3, H, W], float32 [0, 1]
    return t.detach().cpu().permute(1, 2, 0).numpy()

def gray_tensor_to_rgb_numpy(t):
    # t: [1, H, W] -> [H, W, 3]
    gray = t.squeeze(0).detach().cpu()
    rgb = gray.repeat(3, 1, 1)
    return rgb.permute(1, 2, 0).numpy()


def draw_superpoint_keypoints(ax, image, pts, title=""):
    """
    image: [H, W] numpy grayscale
    pts: [3, N] from MagicLeap SuperPointFrontend.run()
         row 0 = x, row 1 = y, row 2 = confidence
    """
    ax.imshow(image, cmap="gray")
    ax.set_title(title)
    ax.axis("off")

    if pts is not None and pts.shape[1] > 0:
        xs = pts[0, :]
        ys = pts[1, :]
        ax.scatter(xs, ys, s=6, facecolors="none", edgecolors="r", linewidths=0.6)


def draw_matched_keypoints(ax, image, matched_xy, title=""):
    ax.imshow(image)
    ax.set_title(title)
    ax.axis("off")

    if matched_xy is not None and len(matched_xy) > 0:
        if torch.is_tensor(matched_xy):
            matched_xy = matched_xy.detach().cpu().numpy()

        ax.scatter(
            matched_xy[:, 0],
            matched_xy[:, 1],
            s=10,
            facecolors="black",
            edgecolors="none",
            linewidths=0.8,
        )


def mutual_nearest_keypoint_matches(fixed_pts, moving_pts, max_dist=5.0):
    """
    fixed_pts:  [3, N] numpy array or torch tensor, rows = x, y, confidence
    moving_pts: [3, M] numpy array or torch tensor

    returns:
        fixed_matched:  [K, 2]
        moving_matched: [K, 2]
        matches:        [K, 2] indices: fixed_idx, moving_idx
    """
    if not torch.is_tensor(fixed_pts):
        fixed_pts = torch.tensor(fixed_pts, dtype=torch.float32)
    if not torch.is_tensor(moving_pts):
        moving_pts = torch.tensor(moving_pts, dtype=torch.float32)

    fixed_xy = fixed_pts[:2, :].T      # [N, 2]
    moving_xy = moving_pts[:2, :].T    # [M, 2]

    if fixed_xy.numel() == 0 or moving_xy.numel() == 0:
        return fixed_xy[:0], moving_xy[:0], torch.empty((0, 2), dtype=torch.long)

    dists = torch.cdist(fixed_xy, moving_xy)  # [N, M]

    fixed_to_moving_dist, fixed_to_moving_idx = dists.min(dim=1)  # [N]
    moving_to_fixed_dist, moving_to_fixed_idx = dists.min(dim=0)  # [M]

    fixed_indices = torch.arange(fixed_xy.shape[0])

    mutual = moving_to_fixed_idx[fixed_to_moving_idx] == fixed_indices
    close_enough = fixed_to_moving_dist <= max_dist

    keep = mutual & close_enough

    matched_fixed_idx = fixed_indices[keep]
    matched_moving_idx = fixed_to_moving_idx[keep]

    fixed_matched = fixed_xy[matched_fixed_idx]
    moving_matched = moving_xy[matched_moving_idx]

    matches = torch.stack([matched_fixed_idx, matched_moving_idx], dim=1)

    return fixed_matched, moving_matched, matches


def meta_value(batch, key, i):
    value = batch["meta"][key][i]
    return value.item() if torch.is_tensor(value) else value


def get_items_by_level(dataset, items_per_level):
    selected = []

    counts = {level: 0 for level in items_per_level}

    for idx in range(len(dataset)):
        sample = dataset[idx]
        level = sample["meta"]["crop_depth"]

        if level in items_per_level and counts[level] < items_per_level[level]:
            selected.append(sample)
            counts[level] += 1

        if all(counts[level] >= items_per_level[level] for level in items_per_level):
            break

    return selected


def visualize_superpoint_by_levels(detector, dataset, items_per_level, max_dist=8.0):
    samples = get_items_by_level(dataset, items_per_level)

    n_items = len(samples)

    fig, axes = plt.subplots(n_items, 2, figsize=(10, 4 * n_items))

    if n_items == 1:
        axes = axes[None, :]

    for i, sample in enumerate(samples):
        fixed = sample["fixed"]
        moving = sample["moving"]

        fixed_vis = sample["fixed_vis"]
        moving_vis = sample["moving_vis"]

        level = sample["meta"]["crop_depth"]

        fixed_gray = tensor_to_numpy_gray(fixed)
        moving_gray = tensor_to_numpy_gray(moving)

        fixed_rgb = tensor_to_numpy_rgb(fixed_vis)
        moving_rgb = tensor_to_numpy_rgb(moving_vis)

        fixed_pts, fixed_desc, fixed_heatmap = detector.detect(fixed_gray)
        moving_pts, moving_desc, moving_heatmap = detector.detect(moving_gray)

        fixed_matched, moving_matched, matches = mutual_nearest_keypoint_matches(
            fixed_pts,
            moving_pts,
            max_dist=max_dist,
        )

        draw_matched_keypoints(
            axes[i, 0],
            fixed_rgb,
            fixed_matched,
            title=f"Fixed | level {level} | {len(fixed_matched)} / {fixed_pts.shape[1]} matched",
        )

        draw_matched_keypoints(
            axes[i, 1],
            moving_rgb,
            moving_matched,
            title=f"Moving | level {level} | {len(moving_matched)} / {moving_pts.shape[1]} matched",
        )

    plt.tight_layout()
    plt.show()

dataset = QuadtreeWSIPairDataset(
    annotation_path=ANNOTATION_PATH,
    input_height=CNN_INPUT_HEIGHT,
    input_width=CNN_INPUT_WIDTH
)

loader = DataLoader(
    dataset,
    batch_size=4,
    shuffle=False,
    num_workers=0,
    pin_memory=True,
)

print("dataset length:", len(dataset))



device = "cuda" if torch.cuda.is_available() else "cpu"

superpoint = SuperPointDetector()


visualize_superpoint_by_levels(
    detector=superpoint,
    dataset=dataset,
    items_per_level={
        0: 1,
        1: 0,
        2: 0,
        3: 0,
        4:5
    
    },
    max_dist=2.0,
)

NameError: name 'keypoint_detectors' is not defined

In [ ]:
def draw_keypoints(ax, image, pts, title=""):
    ax.imshow(image)
    ax.set_title(title)
    ax.axis("off")

    if pts is not None and pts.shape[1] > 0:
        ax.scatter(
            pts[0, :],
            pts[1, :],
            s=8,
            facecolors="none",
            edgecolors="r",
            linewidths=0.6,
        )

def visualize_detector_by_levels(detector, dataset, items_per_level):
    samples = get_items_by_level(dataset, items_per_level)

    n_items = len(samples)
    fig, axes = plt.subplots(n_items, 2, figsize=(10, 4 * n_items))

    if n_items == 1:
        axes = axes[None, :]

    for i, sample in enumerate(samples):
        fixed_gray = tensor_to_numpy_gray(sample["fixed"])
        moving_gray = tensor_to_numpy_gray(sample["moving"])

        fixed_rgb = tensor_to_numpy_rgb(sample["fixed_vis"])
        moving_rgb = tensor_to_numpy_rgb(sample["moving_vis"])

        level = sample["meta"]["crop_depth"]

        fixed_pts = detector.detect(fixed_gray)
        moving_pts = detector.detect(moving_gray)

        draw_keypoints(
            axes[i, 0],
            fixed_rgb,
            fixed_pts,
            title=f"Fixed | level {level} | {fixed_pts.shape[1]} {detector.name} keypoints",
        )

        draw_keypoints(
            axes[i, 1],
            moving_rgb,
            moving_pts,
            title=f"Moving | level {level} | {moving_pts.shape[1]} {detector.name} keypoints",
        )

    plt.tight_layout()
    plt.show()

In [ ]:


detector = SuperPointDetector(threshold=20)

visualize_detector_by_levels(
    detector=detector,
    dataset=dataset,
    items_per_level={
        0: 1,
        1: 0,
        2: 0,
        3: 0,
        4: 5,
    },
)

TypeError: SuperPointDetector.__init__() got an unexpected keyword argument 'threshold'